# Radiology Impression Parser — Overall Normal/Abnormal Classifier

Classifies free-text Bahasa-Indonesia + Latin/English radiology impressions as
**normal** or **abnormal**, scored against the human `rad_label` ground truth.

Pipeline: **load gold → leakage-free train/dev/test split → rule-based classify → evaluate → error analysis → baseline comparison → generalization test (vocabulary-held-out)**.

In-distribution the rules hit 100% — but that is on a *closed* synthetic vocabulary. The honest question is generalization to **unseen findings**, measured in §9 and compared against trained models (TF-IDF, IndoBERT) in `model_trained/`.

Logic lives in `parser.py` (`classify_overall`, `configure_vocabulary`), `data_utils.py` (loading + split), and `validate_generalization.py` (`heldout_split`).

In [1]:
import os, sys
# Run from rule_based/. Put this dir FIRST so local parser.py shadows Python
# 3.9's deprecated stdlib `parser` module; add shared/ for data_utils.
_HERE = os.getcwd()
sys.path[:0] = [_HERE, os.path.join(_HERE, '..', 'shared')]
os.makedirs('outputs', exist_ok=True)
import importlib
import pandas as pd
import data_utils as D
import parser as P
importlib.reload(D); importlib.reload(P)   # pick up edits to the .py files

<module 'parser' from '/Users/reg/Desktop/Internship Dashlabs/rule_based/parser.py'>

## 1. Load the labeled gold set
All 13 `client_syn_*` files; keep rows with impression text **and** a `rad_label` (`normal`/`abnormal`).

In [2]:
gold = D.load_gold()
print('gold rows:', len(gold), '| distinct texts:', gold['text'].nunique())
print('\nclass balance:')
print(gold['rad_label'].value_counts())
print('\nper-client labeled rows:')
print(gold['client'].value_counts().sort_index())

gold rows: 679 | distinct texts: 607

class balance:
rad_label
normal      405
abnormal    274
Name: count, dtype: int64

per-client labeled rows:
client
01     33
02    129
03     23
04     27
05     37
06     52
07     31
08     36
09     30
10     29
11     38
12     31
13    183
Name: count, dtype: int64


## 2. Train / dev / test split on *distinct texts*
Splitting on text (not rows) guarantees the same impression never appears in two splits — so the test number reflects generalization to **unseen** phrasings.

In [3]:
g = D.split_by_text(gold, seed=42)

# leakage check
sets = {s: set(g[g.split == s].text) for s in ['train', 'dev', 'test']}
assert not (sets['train'] & sets['dev']), 'train/dev text overlap!'
assert not (sets['train'] & sets['test']), 'train/test text overlap!'
assert not (sets['dev'] & sets['test']), 'dev/test text overlap!'
print('no text overlap across splits ✓')
print('\nrows per split:')
print(g['split'].value_counts())

no text overlap across splits ✓

rows per split:
split
train    413
dev      136
test     130
Name: count, dtype: int64


## 3. Classify and evaluate
`classify_overall` marks an impression **abnormal** if any clause has an *un-negated* abnormal finding cue; otherwise **normal** ("abnormal wins" on mixed sentences). Positive class for P/R/F1 is `abnormal`.

In [4]:
g['pred'] = g['text'].map(P.classify_overall)

metrics = {}
for s in ['train', 'dev', 'test']:
    sub = g[g.split == s]
    metrics[s] = P.evaluate(sub['rad_label'], sub['pred'], title=f'NEW classifier — {s}')
    print()

=== NEW classifier — train (n=413) ===
accuracy=1.000  precision=1.000  recall=1.000  f1=1.000
confusion: TP=165 FP=0 FN=0 TN=248  (positive='abnormal')

=== NEW classifier — dev (n=136) ===
accuracy=1.000  precision=1.000  recall=1.000  f1=1.000
confusion: TP=57 FP=0 FN=0 TN=79  (positive='abnormal')

=== NEW classifier — test (n=130) ===
accuracy=1.000  precision=1.000  recall=1.000  f1=1.000
confusion: TP=52 FP=0 FN=0 TN=78  (positive='abnormal')



### In-distribution test accuracy (closed synthetic vocabulary)

This 100% is **not** the generalization number — the test split shares its vocabulary with training (the data is synthetic from a fixed phrase set). It shows the method is *correct*, not that it handles unseen findings. The honest generalization test is §9.

In [5]:
print(f"TEST accuracy: {metrics['test']['accuracy']:.1%}  |  F1: {metrics['test']['f1']:.3f}")

TEST accuracy: 100.0%  |  F1: 1.000


## 4. Error analysis (dev set)
Drives the next round of lexicon edits. Keep the **test** set untouched while iterating.

In [6]:
dev = g[g.split == 'dev']
_ = P.evaluate(dev['rad_label'], dev['pred'], title='dev errors',
               show_errors=True, texts=dev['text'])

=== dev errors (n=136) ===
accuracy=1.000  precision=1.000  recall=1.000  f1=1.000
confusion: TP=57 FP=0 FN=0 TN=79  (positive='abnormal')
-- 0 misclassified --


## 5. Baseline comparison — the improvement
Old heuristic: label **normal** if the text mentions a normal keyword anywhere. It misses abnormal findings buried in otherwise-normal sentences.

In [7]:
test = g[g.split == 'test']
print('--- OLD baseline (naive keyword) on TEST ---')
base = P.evaluate(test['rad_label'], test['text'].map(P.classify_naive))
print('\n--- NEW classifier on TEST ---')
new = P.evaluate(test['rad_label'], test['pred'])
print(f"\naccuracy: {base['accuracy']:.1%}  →  {new['accuracy']:.1%}")

--- OLD baseline (naive keyword) on TEST ---
accuracy=0.638  precision=0.667  recall=0.192  f1=0.299
confusion: TP=10 FP=5 FN=42 TN=73  (positive='abnormal')

--- NEW classifier on TEST ---
accuracy=1.000  precision=1.000  recall=1.000  f1=1.000
confusion: TP=52 FP=0 FN=0 TN=78  (positive='abnormal')

accuracy: 63.8%  →  100.0%


## 6. Save predictions for review

In [8]:
out = g.copy()
out['correct'] = out['pred'] == out['rad_label']
out[['client', 'split', 'text', 'rad_label', 'pred', 'correct']].to_csv('outputs/predictions.csv', index=False)
print('wrote outputs/predictions.csv:', len(out), 'rows |', f"{out['correct'].mean():.1%} correct overall")

wrote outputs/predictions.csv: 679 rows | 100.0% correct overall


## 7. Organ-level structured findings
Beyond the overall label, `parse_organs` extracts a per-organ status
(`normal` / `abnormal` / `not_mentioned`) for the 6 tracked organs — the
structured output that can be queried and trended. As a check, deriving the
overall label from the organ findings (abnormal if *any* organ is abnormal)
should reproduce the gold `rad_label`.

In [9]:
from collections import Counter

organ_status = gold['text'].map(P.parse_organs)
mentions = Counter()
per_organ = {o: Counter() for o in P.ORGANS}
for st in organ_status:
    for o in P.ORGANS:
        per_organ[o][st[o]] += 1
        if st[o] != 'not_mentioned':
            mentions[o] += 1

print('Organ mentions across impressions (chart data):')
for o, n in mentions.most_common():
    print(f'  {o:8s} {n}')
print('\nPer-organ status distribution:')
for o in P.ORGANS:
    print(f'  {o:8s} {dict(per_organ[o])}')

# consistency: overall derived from organ-level vs gold
derived = gold['text'].map(P.classify_overall_from_organs)
print()
P.evaluate(gold['rad_label'], derived, title='overall DERIVED from organ parse (full corpus)')

# example structured output
ex = 'Cor membesar, ginjal kanan dan kiri normal, gallbladder unremarkable.'
print('\nexample:', ex)
print('parsed :', P.parse_organs(ex))

Organ mentions across impressions (chart data):
  hepar    325
  pulmo    314
  ginjal   295
  vesica   284
  cor      273
  pleura   230

Per-organ status distribution:
  cor      {'not_mentioned': 406, 'normal': 221, 'abnormal': 52}
  pulmo    {'normal': 259, 'not_mentioned': 365, 'abnormal': 55}
  hepar    {'normal': 268, 'not_mentioned': 354, 'abnormal': 57}
  ginjal   {'not_mentioned': 384, 'normal': 248, 'abnormal': 47}
  vesica   {'normal': 235, 'not_mentioned': 395, 'abnormal': 49}
  pleura   {'not_mentioned': 449, 'normal': 175, 'abnormal': 55}

=== overall DERIVED from organ parse (full corpus) (n=679) ===
accuracy=1.000  precision=1.000  recall=1.000  f1=1.000
confusion: TP=274 FP=0 FN=0 TN=405  (positive='abnormal')

example: Cor membesar, ginjal kanan dan kiri normal, gallbladder unremarkable.
parsed : {'cor': 'abnormal', 'pulmo': 'not_mentioned', 'hepar': 'not_mentioned', 'ginjal': 'normal', 'vesica': 'normal', 'pleura': 'not_mentioned'}


## 8. Spot-check decisive cases

In [10]:
cases = [
    ('Cor membesar, ginjal kanan dan kiri normal, gallbladder unremarkable.', 'abnormal'),
    ('Cor tidak membesar; pulmo dalam batas normal.', 'normal'),
    ('Foto thorax: no hepatomegaly; no active lung lesion; cardiomegaly; batu ginjal kiri.', 'abnormal'),
    ('Cardiomegaly dengan CTR 0.58.', 'abnormal'),
    ('Foto thorax: CTR normal; ginjal dalam batas normal.', 'normal'),
    ('Hasil: lungs clear. efusi pleura.', 'abnormal'),
    ('Kesan: cholecystitis. liver unremarkable.', 'abnormal'),
    ('Enlarged cardiac silhouette; pleura normal; tidak tampak batu empedu.', 'abnormal'),
]
for t, exp in cases:
    got = P.classify_overall(t)
    print(('OK ' if got == exp else 'XX '), f'exp={exp:8s} got={got:8s} | {t}')

OK  exp=abnormal got=abnormal | Cor membesar, ginjal kanan dan kiri normal, gallbladder unremarkable.
OK  exp=normal   got=normal   | Cor tidak membesar; pulmo dalam batas normal.
OK  exp=abnormal got=abnormal | Foto thorax: no hepatomegaly; no active lung lesion; cardiomegaly; batu ginjal kiri.
OK  exp=abnormal got=abnormal | Cardiomegaly dengan CTR 0.58.
OK  exp=normal   got=normal   | Foto thorax: CTR normal; ginjal dalam batas normal.
OK  exp=abnormal got=abnormal | Hasil: lungs clear. efusi pleura.
OK  exp=abnormal got=abnormal | Kesan: cholecystitis. liver unremarkable.
OK  exp=abnormal got=abnormal | Enlarged cardiac silhouette; pleura normal; tidak tampak batu empedu.


## 9. Generalization to unseen findings — the honest test

The 100% above is on a *closed* vocabulary. To measure what happens on a genuinely **unseen** finding, hold two whole finding families — **effusion + stones** (10 cues) — out of the rule vocabulary (and, in `model_trained/`, out of the trained models' data too), then score on the impressions that use them.

The key subset is **sole-signal** abnormals: impressions where the held-out cue is the *only* abnormal signal. A rule system cannot catch these once the cue is unseen — it has nothing left to match. A trained model can still recover some from surrounding context. That contrast is the real story.

In [11]:
import validate_generalization as V
importlib.reload(V)

# effusion + stones held out. heldout_split marks the sole-signal abnormals.
train_df, eval_df = V.heldout_split(gold)
sole = eval_df[eval_df['sole_signal']]
n_abn = int((eval_df['rad_label'] == 'abnormal').sum())
print('held-out cues:', V.HELDOUT_CUES)
print(f'eval set: {len(eval_df)} impressions ({n_abn} abnormal, '
      f'{len(eval_df) - n_abn} normal) | sole-signal abnormals: {len(sole)}')

# simulate the cues being NEVER seen, score, then restore the full vocabulary
P.configure_vocabulary(exclude_cues=V.HELDOUT_CUES)
ho_acc = (eval_df['text'].map(P.classify_overall).values == eval_df['rad_label'].values).mean()
sole_recall = (sole['text'].map(P.classify_overall) == 'abnormal').mean()
P.reset_vocabulary()
assert (gold['text'].map(P.classify_overall).values == gold['rad_label'].values).mean() == 1.0, \
    'vocabulary not fully restored!'

print(f'\nRULE held-out accuracy   : {ho_acc:.1%}')
print(f'RULE sole-signal recall  : {sole_recall:.1%}   <- rules cannot match vocabulary they never saw')
print('\nTF-IDF + IndoBERT are scored on the SAME held-out set in model_trained/;')
print('the 3-way contrast lives in model_trained/outputs/generalization.json.')

held-out cues: ['efusi', 'effusion', 'lithiasis', 'litiasis', 'kolelitiasis', 'nephrolithiasis', 'batu empedu', 'batu ginjal', 'gallstones?', 'kidney stones?']
eval set: 274 impressions (148 abnormal, 126 normal) | sole-signal abnormals: 40

RULE held-out accuracy   : 69.3%
RULE sole-signal recall  : 0.0%   <- rules cannot match vocabulary they never saw

TF-IDF + IndoBERT are scored on the SAME held-out set in model_trained/;
the 3-way contrast lives in model_trained/outputs/generalization.json.


### Takeaway

Two numbers, honestly labeled:

- **In-distribution (closed vocabulary): 100%** — the method is correct; the negation-aware clause logic lifts a naive keyword baseline from 63.8% to 100%.
- **Unseen findings (sole-signal held-out): 0%** — a hand-written rule set cannot generalize to vocabulary it never saw. On the same held-out set a trained TF-IDF model recovers ~58%, and a fine-tuned IndoBERT is expected higher (see `model_trained/`).

The headline is the **contrast**, not the 100%: rules win in-distribution, learned models win on unseen findings — the tradeoff made explicit.